# Synthetic Data Validation: Geometric Diagnostics vs Known True k₀

**Goal**: Verify that the 9 empirical findings from the real-data report reproduce
when we control the true factor rank k₀.

**Findings to validate**:
1. Sharpe ↑ in P (saturates ~4096) — KMZ Theorem 1
2. OOS R² ↑ in P (benign overfitting)
3. Spectral gap cliff at k₀ — Theorem 6.1(iii)
4. Low drift anisotropy ρ_θ → high Sharpe — Theorem 6.1(i)
5. All metrics ↑ in z — ridge–curvature chain
6. z=0 never finds optimal k — flat Hessian
7. d_proj ↓ in k at z=0 — Theorem 6.1(i)
8. Moderate d_proj is optimal (Goldilocks zone)
9. High Sharpe with erank collapse at k > k₀

In [4]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# Add the project root so "from src.xxx" imports work
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.rff import RandomFourierFeatures
from src.generate_synthetic_panel import generate_synthetic_panel
from src._grass_worker import (
    build_rff_inputs, build_jobs, run_sweep, aggregate, make_results_df,
    _z_label,
)

print('Imports OK')

Imports OK


## 1. Generate synthetic panel with known true k₀ = 5

In [8]:
TRUE_K0 = 12

df_synth, truth, char_cols = generate_synthetic_panel(
    T   = 360,
    N   = 50,
    m   = 30,            # bump up to give enough characteristics for k₀=12
    k0  = TRUE_K0,
    seed = 42,
    # 12 signal eigenvalues with clear decay above noise floor (σ²=1)
    signal_eigenvalues = np.array([
        8.0, 6.0, 4.5, 3.5, 3.0, 2.5, 2.0, 1.8, 1.6, 1.4, 1.2, 1.05
    ]),
    sigma_eps      = 1.0,
    factor_ar1     = 0.5,
    factor_vol     = 0.3,
    w_drift_scale  = 0.02,
    z_corr         = 0.3,
    hetero_strength = 0.3,
)

# Grid mirrors real-data report exactly
NUM_FACTORS_LIST = [4, 8, 12, 16, 20, 24, 28, 32]

## 2. RFF expansion of characteristics

In [9]:
# ===========================================================================
# GRID — sweep over k, P, z to validate all 9 findings
# ===========================================================================
NUM_FACTORS_LIST = [2, 4, 5, 8, 12, 16, 20, 24]   # spans below/at/above true k₀=5
Z_VALUES         = [0, 10, 50]                      # illusory + virtuous
N_FEATURES_RFF   = [64, 256, 1024, 4096]            # P-scaling
NUM_ITER_RFF     = 2                                # RFF seeds (keep low for speed)
WINDOW_LEN       = 24                               # matches real-data backtest
GAMMA            = 0.25

# Dense char matrix for RFF
X_chars = (
    df_synth[char_cols]
    .apply(lambda s: s.fillna(s.mean()), axis=0)  # impute
    .astype(np.float64)
)

print(f"X_chars shape : {X_chars.shape}")

X_chars shape : (18000, 30)


## 3. Run backtest sweep

In [ ]:
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df_synth,
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)
print(f"  {len(jobs)} jobs queued.")

raw_results = run_sweep(jobs, verbose=True)
results_agg = aggregate(raw_results)
df_results  = make_results_df(results_agg)

print(f"\nResults shape: {df_results.shape}")
df_results.head(10)

Pre-computing RFF features ...
  8 RFF datasets ready.
  192 jobs queued.
Dispatching 192 jobs across 7 workers (cost range 181–481589, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done  13 out of 192 | elapsed: 676.5min remaining: 9315.1min


## 4. Compute drift anisotropy ρ_θ from raw results

Anisotropy ρ_θ = mean(θ) / max(θ) — the strongest geometric predictor in real data (r = −0.52).

In [ ]:
# Add anisotropy to the results DataFrame
aniso_rows = []
for key, val in results_agg.items():
    k, P, z_lab = key
    # avg across seeds
    all_aniso = []
    for pa_series in val['principal_angles_series']:
        for angles in pa_series:
            if len(angles) > 0 and angles[0] > 1e-8:
                all_aniso.append(np.mean(angles) / angles[0])
    mean_aniso = np.mean(all_aniso) if all_aniso else np.nan
    aniso_rows.append({'k': k, 'P': P, 'z': z_lab, 'anisotropy': mean_aniso})

df_aniso = pd.DataFrame(aniso_rows).set_index(['k', 'P', 'z'])
df_results = df_results.join(df_aniso)
print("Anisotropy column added.")
df_results[['avg_sharpe', 'avg_spectral_gap', 'avg_erank', 'anisotropy']].head(15)

---
## 5. Validation Plots

### Finding 3: Spectral gap cliff at true k₀ = 5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
P_fixed = 1024
for z_val in Z_VALUES:
    z_lab = _z_label(z_val)
    sub = df_results.xs((P_fixed, z_lab), level=('P', 'z'))
    ax.plot(sub.index, sub['avg_spectral_gap'], 'o-', label=f'z={z_val}')

ax.axvspan(TRUE_K0 - 0.5, TRUE_K0 + 0.5, alpha=0.2, color='orange', label=f'true k₀={TRUE_K0}')
ax.set_xlabel('Factors k')
ax.set_ylabel('Spectral gap λ_k/λ_1')
ax.set_title(f'Finding 3: Spectral Gap Cliff (P={P_fixed}, true k₀={TRUE_K0})')
ax.legend()
plt.tight_layout()
plt.show()

### Finding 1 & 2: Virtue of Complexity — Sharpe & R² increase with P

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for k_val in [TRUE_K0, 2 * TRUE_K0]:
    for z_val in [10, 50]:
        z_lab = _z_label(z_val)
        try:
            sub = df_results.xs((z_lab,), level=('z',))
            sub = sub.xs(k_val, level='k')
            axes[0].plot(sub.index, sub['avg_sharpe'], 'o-',
                         label=f'k={k_val}, z={z_val}')
            axes[1].plot(sub.index, sub['avg_r2_oos'], 'o-',
                         label=f'k={k_val}, z={z_val}')
        except KeyError:
            pass

axes[0].set_xlabel('RFF dimension P'); axes[0].set_ylabel('Annualised Sharpe')
axes[0].set_title('Finding 1: Sharpe ↑ with P'); axes[0].legend()
axes[0].set_xscale('log')

axes[1].set_xlabel('RFF dimension P'); axes[1].set_ylabel('OOS R²')
axes[1].set_title('Finding 2: R² ↑ with P'); axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

### Finding 5: All metrics co-move with z

In [ ]:
k_plot = TRUE_K0
P_plot = 1024

metrics_z = []
for z_val in Z_VALUES:
    z_lab = _z_label(z_val)
    try:
        row = df_results.loc[(k_plot, P_plot, z_lab)]
        metrics_z.append({
            'z': z_val,
            'Sharpe': row['avg_sharpe'],
            'd_proj': row['avg_subspace_stability'],
            'θ_max': row['avg_max_principal_angle'],
            'erank': row['avg_erank'],
        })
    except KeyError:
        pass

df_z = pd.DataFrame(metrics_z).set_index('z')
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(df_z.index, df_z['d_proj'], 'o-b', label='d_proj')
ax1.plot(df_z.index, df_z['θ_max'], 's--r', label='θ_max')
ax2.plot(df_z.index, df_z['Sharpe'], 'D-g', label='Sharpe')

ax1.set_xlabel('Regularisation z')
ax1.set_ylabel('Geometric metrics')
ax2.set_ylabel('Sharpe')
ax1.set_title(f'Finding 5: All Metrics Co-Move with z (k={k_plot}, P={P_plot})')
ax1.legend(loc='upper left'); ax2.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Finding 4: Drift anisotropy vs Sharpe

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Virtuous regime only (z > 0)
mask_virt = ~df_results.index.get_level_values('z').str.contains('0e\\+00')
virt = df_results[mask_virt].dropna(subset=['anisotropy'])
illu = df_results[~mask_virt].dropna(subset=['anisotropy'])

ax.scatter(virt['anisotropy'], virt['avg_sharpe'], c='steelblue',
           label='Virtuous (z>0)', alpha=0.7)
ax.scatter(illu['anisotropy'], illu['avg_sharpe'], c='red', marker='^',
           label='Illusory (z=0)', alpha=0.7)

# Correlation within virtuous regime
if len(virt) > 3:
    r = virt[['anisotropy', 'avg_sharpe']].corr().iloc[0, 1]
    ax.set_title(f'Finding 4: Anisotropy vs Sharpe (r_virt = {r:.2f})')
else:
    ax.set_title('Finding 4: Anisotropy vs Sharpe')

ax.set_xlabel('Drift anisotropy ρ_θ = mean(θ)/max(θ)')
ax.set_ylabel('Annualised Sharpe')
ax.legend()
plt.tight_layout()
plt.show()

### Finding 7: d_proj decays with k at z=0

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
z_lab = _z_label(0)
P_fixed = 1024

try:
    sub = df_results.xs((P_fixed, z_lab), level=('P', 'z'))
    ax.plot(sub.index, sub['avg_subspace_stability'], 'o-b', label='d_proj')
    ax.set_xlabel('Factors k')
    ax.set_ylabel('Mean d_proj')
    ax.axvline(TRUE_K0, ls='--', c='orange', label=f'true k₀={TRUE_K0}')
    ax.set_title(f'Finding 7: d_proj Decays with k at z=0 (P={P_fixed})')
    ax.legend()
except KeyError:
    print('No z=0 data at this P.')

plt.tight_layout()
plt.show()

### Finding 6: z=0 never finds optimal k  vs  z>0 clean peaks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: z=0 erratic
z_lab0 = _z_label(0)
for P_val in N_FEATURES_RFF:
    try:
        sub = df_results.xs((P_val, z_lab0), level=('P', 'z'))
        axes[0].plot(sub.index, sub['avg_sharpe'], 'o-', label=f'P={P_val}')
    except KeyError:
        pass
axes[0].set_title('z=0 (no ridge) — erratic')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Sharpe')
axes[0].axvline(TRUE_K0, ls='--', c='orange')
axes[0].legend()

# Right: z>0 clean
for P_val in N_FEATURES_RFF:
    sharpes_avg = []
    ks = []
    for k_val in NUM_FACTORS_LIST:
        vals = []
        for z_val in [z for z in Z_VALUES if z > 0]:
            try:
                row = df_results.loc[(k_val, P_val, _z_label(z_val))]
                vals.append(row['avg_sharpe'])
            except KeyError:
                pass
        if vals:
            sharpes_avg.append(np.mean(vals))
            ks.append(k_val)
    if ks:
        axes[1].plot(ks, sharpes_avg, 'o-', label=f'P={P_val}')

axes[1].set_title('z≥10 (with ridge) — clean peaks')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Sharpe (avg z≥10)')
axes[1].axvline(TRUE_K0, ls='--', c='orange')
axes[1].legend()

plt.suptitle(f'Finding 6: z=0 Never Finds Optimal k (true k₀={TRUE_K0})', y=1.02)
plt.tight_layout()
plt.show()

### Finding 8: Goldilocks zone — Sharpe vs d_proj

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    df_results['avg_subspace_stability'],
    df_results['avg_sharpe'],
    c=np.log10(df_results.index.get_level_values('P').astype(float)),
    cmap='viridis', alpha=0.7, edgecolors='k', linewidths=0.3,
)
plt.colorbar(sc, label='log₁₀(P)')
ax.set_xlabel('d_proj (subspace stability)')
ax.set_ylabel('Annualised Sharpe')
ax.set_title('Finding 8: Goldilocks Zone')
plt.tight_layout()
plt.show()

### Finding 9: erank collapse at k > k₀ with high Sharpe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

P_fixed = 1024
for z_val in [z for z in Z_VALUES if z > 0]:
    z_lab = _z_label(z_val)
    try:
        sub = df_results.xs((P_fixed, z_lab), level=('P', 'z'))
        axes[0].plot(sub.index, sub['avg_erank'], 'o-', label=f'z={z_val}')
        axes[1].plot(sub.index, sub['avg_sharpe'], 'o-', label=f'z={z_val}')
    except KeyError:
        pass

axes[0].axvline(TRUE_K0, ls='--', c='orange', label=f'true k₀={TRUE_K0}')
axes[0].plot(NUM_FACTORS_LIST, NUM_FACTORS_LIST, 'k--', alpha=0.3, label='erank=k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('erank(Σ̂_f)')
axes[0].set_title('erank vs k'); axes[0].legend()

axes[1].axvline(TRUE_K0, ls='--', c='orange', label=f'true k₀={TRUE_K0}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Sharpe')
axes[1].set_title('Sharpe vs k'); axes[1].legend()

plt.suptitle(f'Finding 9: erank Collapse at k > k₀ (P={P_fixed})', y=1.02)
plt.tight_layout()
plt.show()

### Matched-pair comparison: z=0 vs z>0 (Finding 2 from report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
P_fixed = 1024
z0_lab = _z_label(0)
z10_lab = _z_label(10)

for metric, ax_i, ylabel in [
    ('avg_subspace_stability', axes[0], 'Mean d_proj'),
    ('avg_sharpe', axes[1], 'Sharpe'),
]:
    for z_lab, color, label in [(z0_lab, 'red', 'z=0'), (z10_lab, 'blue', 'z=10')]:
        try:
            sub = df_results.xs((P_fixed, z_lab), level=('P', 'z'))
            ax_i.bar(sub.index + (0.3 if z_lab == z10_lab else 0), sub[metric],
                     width=0.3, color=color, alpha=0.7, label=label)
        except KeyError:
            pass
    ax_i.axvline(TRUE_K0, ls='--', c='orange')
    ax_i.set_xlabel('k'); ax_i.set_ylabel(ylabel); ax_i.legend()

plt.suptitle(f'Matched Pairs (P={P_fixed}): Illusory vs Virtuous', y=1.02)
plt.tight_layout()
plt.show()

## 6. Summary correlation table

In [ ]:
# Correlation of each metric with Sharpe, split by regime
cols = ['avg_subspace_stability', 'avg_max_principal_angle',
        'avg_mean_principal_angle', 'avg_erank', 'avg_spectral_gap', 'anisotropy']

mask_virt = ~df_results.index.get_level_values('z').str.contains('0e\\+00')

corr_all  = df_results[cols + ['avg_sharpe']].corr()['avg_sharpe'].drop('avg_sharpe')
corr_virt = df_results.loc[mask_virt, cols + ['avg_sharpe']].corr()['avg_sharpe'].drop('avg_sharpe')

df_corr = pd.DataFrame({'All data': corr_all, 'Virtuous (z>0)': corr_virt})
print("\n=== Metric–Sharpe Correlations ===")
print(df_corr.round(3))

## 7. Full results table for paper Section 8 (Simulated Data Analysis)

In [ ]:
# Top configs by Sharpe
top = df_results.nlargest(10, 'avg_sharpe')[[
    'avg_sharpe', 'avg_spectral_gap', 'avg_erank', 'avg_subspace_stability', 'anisotropy'
]]
print("\n=== Top 10 Configurations by Sharpe ===")
print(top.round(4))